# Jigsaw - Agile Community Rules Classification
Using AI models to help moderators uphold community-specific norms.

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

jigsaw_agile_community_rules_path = kagglehub.competition_download('jigsaw-agile-community-rules')
siddharth050_bart_fine_tuned_transformers_bart_cnn_20k_1_path = kagglehub.model_download('siddharth050/bart-fine-tuned/Transformers/bart_cnn_20k/1')
jonathanchan_baai_transformers_bge_base_en_v1_5_1_path = kagglehub.model_download('jonathanchan/baai/Transformers/bge-base-en-v1.5/1')

print('Data source import complete.')


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, AutoModelForSequenceClassification
import pandas as pd
import re
import numpy as np
import random
from tqdm import tqdm
from heapq import heappush, heappop
from datasets import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

seed = 42
random.seed(seed)

2025-10-06 14:47:53.599868: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759762073.985231      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759762074.083653      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


cuda


In [ ]:
model_name = "/kaggle/input/baai/transformers/bge-base-en-v1.5/1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /kaggle/input/baai/transformers/bge-base-en-v1.5/1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
df=pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/train.csv")

def build_input(row):
    return f"""
                Rule: {row['rule']}
                Comment: {row['body']}

                Positive Examples:
                - {row['positive_example_1']}
                - {row['positive_example_2']}

                Negative Examples:
                - {row['negative_example_1']}
                - {row['negative_example_2']}
            """

df['text'] = df.apply(build_input, axis=1)
df['labels'] = df['rule_violation'].astype(int)

train_dataset = Dataset.from_pandas(df[['text', 'labels']])

def tokenize_fn(examples):
    tokenized = tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512,
    )
    tokenized["labels"] = examples["labels"]
    return tokenized

train_dataset = train_dataset.map(tokenize_fn, batched=True)
train_dataset = train_dataset.remove_columns(['text'])

print(np.array(train_dataset['input_ids']).shape)
print(np.array(train_dataset['labels']).shape)

Map:   0%|          | 0/2029 [00:00<?, ? examples/s]

(2029, 512)
(2029,)


In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    gradient_accumulation_steps=4,
    num_train_epochs=4,
    logging_steps=10,
    disable_tqdm=False,
    report_to="none",
    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
10,0.676500
20,0.667700
30,0.601100
40,0.507800
50,0.422200
60,0.437900
70,0.370400
80,0.279700
90,0.308700
100,0.245000


TrainOutput(global_step=128, training_loss=0.3994527431204915, metrics={'train_runtime': 434.866, 'train_samples_per_second': 18.663, 'train_steps_per_second': 0.294, 'total_flos': 2135409325301760.0, 'train_loss': 0.3994527431204915, 'epoch': 4.0})

In [ ]:
df=pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/test.csv")

df['text'] = df.apply(build_input, axis=1)

test_dataset = Dataset.from_pandas(df[['text']])
def tokenize_fn(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512,
    )

test_dataset = test_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.remove_columns(['text'])

print(np.array(test_dataset['input_ids']).shape)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

(10, 512)


In [ ]:
from torch.utils.data import DataLoader
from transformers import default_data_collator

dataloader = DataLoader(
    test_dataset,
    batch_size=8,
    collate_fn=default_data_collator
)

model.eval()
preds = []

with torch.no_grad():
    for batch in dataloader:
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)

        if outputs.logits.ndim == 3:
            logits = outputs.logits[:, -1, :]
        else:
            logits = outputs.logits

        pred = torch.argmax(logits, dim=-1)
        preds.extend(pred.cpu().numpy())

print(preds)

[0, 0, 1, 1, 1, 0, 1, 0, 0, 1]


In [ ]:
submission = pd.DataFrame({
    "row_id": df["row_id"],
    "rule_violation": preds
})
submission.to_csv("submission.csv", index=False)